# Build a full CLIP release

Set `release_name` to the full release that already contains the matching BEiT-3 index and `frames.csv`. This notebook downloads organizer CLIP features to Colab SSD and writes only `clip.faiss` into that release.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

!pip -q install faiss-cpu
!mkdir -p /content/work
!wget -q https://aic-data.ledo.io.vn/clip-features-32-aic25-b1.zip -O /content/work/clip-features.zip
!unzip -q /content/work/clip-features.zip -d /content/work/clip-features

In [ ]:
import csv
import shutil
from pathlib import Path

import faiss
import numpy as np

release_name = "full-v1"
work = Path("/content/work")
root = Path("/content/drive/MyDrive/AIC_ARTIFACTS")
artifacts = root / "releases" / release_name

if not (artifacts / "beit3.faiss").exists() or not (artifacts / "frames.csv").exists():
    raise ValueError(f"{release_name} needs beit3.faiss and frames.csv before building CLIP")

In [ ]:
with (artifacts / "frames.csv").open(newline="", encoding="utf-8") as file:
    frame_rows = list(csv.DictReader(file))

rows_by_video = {}
for row in frame_rows:
    rows_by_video.setdefault(row["video_id"], []).append(row)

for rows in rows_by_video.values():
    rows.sort(key=lambda row: int(row["frame_id"]))
    keyframe_numbers = [int(row["keyframe_n"]) for row in rows]
    if keyframe_numbers != sorted(keyframe_numbers):
        raise ValueError(f"{rows[0]['video_id']}: frames.csv is not in keyframe order")

index = None
indexed = 0
for feature_path in sorted(work.rglob("L*_V*.npy")):
    video_id = feature_path.stem
    rows = rows_by_video[video_id]
    vectors = np.load(feature_path, allow_pickle=False).astype("float32")
    if vectors.ndim != 2 or len(vectors) != len(rows):
        raise ValueError(f"{video_id}: feature/catalog mismatch")
    if not np.isfinite(vectors).all():
        raise ValueError(f"{video_id}: invalid CLIP vectors")
    faiss.normalize_L2(vectors)
    if index is None:
        index = faiss.IndexIDMap2(faiss.IndexFlatIP(vectors.shape[1]))
    ids = np.array([int(row["frame_id"]) for row in rows], dtype="int64")
    index.add_with_ids(vectors, ids)
    indexed += len(vectors)

if index is None or indexed != len(frame_rows):
    raise ValueError(f"indexed {indexed} CLIP vectors for {len(frame_rows)} catalog rows")
faiss.write_index(index, str(artifacts / "clip.faiss"))
print(f"published {release_name}/clip.faiss with {index.ntotal} keyframes")
shutil.rmtree(work)

In [ ]:
clip_index = faiss.read_index(str(artifacts / "clip.faiss"))
beit3_index = faiss.read_index(str(artifacts / "beit3.faiss"))
assert clip_index.ntotal == beit3_index.ntotal == len(frame_rows)
print(f"{release_name}: {clip_index.ntotal} CLIP vectors and {beit3_index.ntotal} BEiT-3 vectors")